##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# EmbeddingGemma2 LiteRT: RAG on Edge

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/tutorials/EmbeddingGemma2_LiteRT_RAG.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

### Overview

This notebook walks through **on-device RAG** over the **Gemma 4 technical report** (`data/gemma4_technical_report.pdf`): parse and chunk the PDF, store embeddings in **Qdrant Edge**, then answer questions using retrieved context. Inference stays local—no API calls at query time.

- **EmbeddingGemma 2** (LiteRT-LM) embeds chunks and queries (768-d).
- **Qdrant Edge** holds the vector index in a local shard (`db_gemma/`).
- **Gemma 4 E2B** (LiteRT-LM) generates short answers from the top retrieved passages.

**Pipeline:** download models → load engines → index the PDF → run RAG queries.

## 0 · Install dependencies

Install the packages used for PDF parsing, Hub downloads, LiteRT inference, vector search, and optional UI.
Run this cell once when you set up a new environment.

In [2]:
%pip install -q edgeparse huggingface-hub litert-lm-api qdrant-edge-py

In [1]:
import nest_asyncio
nest_asyncio.apply()

## 1 · Download models and assets

Download the LiteRT embedding and generation checkpoints from Hugging Face.

> Models used:

Fetch EmbeddingGemma 2 and Gemma 4 E2B as `.litertlm` files into `models/`.
The next cell downloads both and checks that the expected paths exist.

In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download

EMBEDDING_REPO_ID = "litert-community/embeddinggemma-2-740m-litert-lm"
EMBEDDING_FILENAME = "embeddinggemma-2-740m.litertlm"
EMBEDDING_LOCAL_DIR = "models/embeddinggemma2"
EMBED_MODEL_PATH = f"{EMBEDDING_LOCAL_DIR}/{EMBEDDING_FILENAME}"

GEMMA_REPO_ID = "litert-community/gemma-4-E2B-it-litert-lm"
GEMMA_FILENAME = "gemma-4-E2B-it.litertlm"
GEMMA_LOCAL_DIR = "models/gemma4"
GEMMA_MODEL_PATH = f"{GEMMA_LOCAL_DIR}/{GEMMA_FILENAME}"

def download_embedding_model() -> str:
    return hf_hub_download(
        repo_id=EMBEDDING_REPO_ID,
        filename=EMBEDDING_FILENAME,
        local_dir=EMBEDDING_LOCAL_DIR,
    )


def download_gemma_litert_model() -> str:
    return hf_hub_download(
        repo_id=GEMMA_REPO_ID,
        filename=GEMMA_FILENAME,
        local_dir=GEMMA_LOCAL_DIR,
    )


download_embedding_model()
download_gemma_litert_model()

### Document

The corpus is the [Gemma 4 technical report](https://arxiv.org/abs/2607.02770) (arXiv:2607.02770), saved as `data/gemma4_technical_report.pdf`.
On Colab or a fresh clone, the next cell downloads the PDF with `curl` before indexing.

In [4]:
!curl -fsSL --create-dirs -o data/gemma4_technical_report.pdf https://arxiv.org/pdf/2607.02770.pdf

In [3]:
DOCUMENT_PATH = "data/gemma4_technical_report.pdf"

## 2 · Load models

Create the LiteRT embedding and generation engines from the files you downloaded.
Both use the GPU backend here; adjust if you run on CPU or another device.

### Embedding engine (EmbeddingGemma 2)

Loads EmbeddingGemma 2 and sets chunk size plus cache directory for inference.
This engine produces 768-d vectors for document chunks and for search queries.

In [4]:
from pathlib import Path
from litert_lm import Backend, EmbeddingEngine

MODEL_DIM = 768
CACHE_DIR = ".cache"
SHARD_PATH = "db_gemma"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 0

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

embedding_engine = EmbeddingEngine(
    EMBED_MODEL_PATH,
    backend=Backend.CPU(),
    cache_dir=CACHE_DIR,
)

### Generation engine (Gemma 4 E2B)

Loads the instruction-tuned Gemma 4 E2B checkpoint for text generation.
Hub access is turned off so answers use only the local model file.

In [5]:
import os

import litert_lm

os.environ["HF_HUB_OFFLINE"] = "1"

generation_engine = litert_lm.Engine(
    GEMMA_MODEL_PATH,
    backend=litert_lm.Backend.CPU(),
)

## 3 · Indexing

Parse the PDF, split it into fixed-size chunks, embed each chunk, and store vectors in a local Qdrant Edge shard.
If `db_gemma` already has data, the shard is reopened instead of rebuilt from scratch.

In [7]:
import edgeparse
from qdrant_edge import (
    Distance,
    EdgeConfig,
    EdgeShard,
    EdgeVectorParams,
    Point,
    UpdateOperation,
)


def load_or_create_shard(shard_path: str) -> EdgeShard:
    config = EdgeConfig(
        vectors=EdgeVectorParams(size=MODEL_DIM, distance=Distance.Cosine)
    )
    path = Path(shard_path)
    path.mkdir(parents=True, exist_ok=True)
    if any(path.iterdir()):
        return EdgeShard.load(shard_path)
    return EdgeShard.create(shard_path, config)


def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    return [
        chunk
        for start in range(0, len(text), chunk_size - overlap)
        if (chunk := text[start : start + chunk_size].strip())
    ]


shard = load_or_create_shard(SHARD_PATH)

markdown = edgeparse.convert(DOCUMENT_PATH, format="markdown")
chunks = chunk_text(markdown, CHUNK_SIZE, CHUNK_OVERLAP)

responses = embedding_engine.compute_embedding_batch(chunks)
embeddings = [response.embedding for response in responses]

shard.update(
    UpdateOperation.upsert_points(
        [
            Point(
                index,
                embeddings[index - 1],
                {
                    "text": chunk,
                    "source": Path(DOCUMENT_PATH).name,
                    "chunk_index": index - 1,
                },
            )
            for index, chunk in enumerate(chunks, start=1)
        ]
    )
)
shard.optimize()
len(chunks)

45

## 4 · RAG

Embed the user question, pull the closest chunks from the shard, and pass them to Gemma 4 with a short system prompt.
The sample queries at the end print one streamed answer each, then close both engines.

In [8]:
from qdrant_edge import Query, QueryRequest


def retrieve_context(user_query: str, limit: int = 2) -> str:
    query_vector = embedding_engine.compute_embedding(user_query).embedding
    results = shard.query(
        QueryRequest(
            query=Query.Nearest(query_vector),
            limit=limit,
            with_payload=True,
        )
    )
    return "".join(point.payload["text"] for point in results)


def system_messages() -> list[litert_lm.Message]:
    return [
        litert_lm.Message.system(
            "You are a Gemma 4 technical report assistant. "
            "Answer the user question using only the provided context. "
            "If the answer is not in the context, say: I don't know"
            "Be smart to understand the context well and do not guess or add extra explanation."
            "Answer in 1-2 short sentences. Do not elaborate unless told by user"
        )
    ]


def answer_stream(user_query: str, limit: int = 3, context: str | None = None):
    if context is None:
        context = retrieve_context(user_query, limit=limit)
    with generation_engine.create_conversation(
        messages=system_messages(),
        sampler_config=litert_lm.SamplerConfig(top_k=1),
    ) as conversation:
        for chunk in conversation.send_message_async(
            f"Context:\n{context}\n\nQuestion:\n{user_query}"
        ):
            for item in chunk.get("content", []):
                if item.get("type") == "text" and item.get("text"):
                    yield item["text"]


QUERIES = [
    "What architecture is Gemma 4 based on?",
    "What context length does Gemma 4 support?",
]

for i, user_query in enumerate(QUERIES, start=1):
    print(f"\n[{i}] {user_query}")
    context = retrieve_context(user_query)
    print("".join(answer_stream(user_query, context=context)))

generation_engine.close()


[1] What architecture is Gemma 4 based on?
Gemma 4 model suite features both dense architectures and a Mixture-of-Experts (MoE) variant.

[2] What context length does Gemma 4 support?
The context mentions that extended contexts lead to a memory explosion in the KV cache, and Gemma 4 conserves a 5:1 ratio of local sliding window to global self-attention (4:1 for the 2.3B model).
